# 🤖 Comprehensive Model Comparison & Evaluation

## Overview
This notebook implements and compares multiple machine learning algorithms for Ethiopian salary prediction. We'll evaluate various models using comprehensive metrics and cross-validation techniques.

### Models to Compare:
- 📈 **Linear Regression**: Baseline linear model
- 🌲 **Random Forest**: Ensemble tree-based model
- 🚀 **Gradient Boosting**: Advanced boosting algorithm
- 🎯 **Support Vector Regression**: Kernel-based approach
- 🧠 **Neural Network**: Multi-layer perceptron
- 🔍 **Ridge/Lasso**: Regularized linear models

### Evaluation Metrics:
- 📊 **RMSE**: Root Mean Squared Error
- 📈 **R² Score**: Coefficient of determination
- 📉 **MAE**: Mean Absolute Error
- 🎯 **MAPE**: Mean Absolute Percentage Error

---

In [ ]:
# Import comprehensive machine learning libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn models and utilities
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, 
    RandomizedSearchCV, KFold, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Regression models
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, 
    BayesianRidge, HuberRegressor
)
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    AdaBoostRegressor, ExtraTreesRegressor
)
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

# Advanced models
try:
    import xgboost as xgb
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost not available. Install with: pip install xgboost")

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("⚠️ LightGBM not available. Install with: pip install lightgbm")

# Evaluation metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, 
    r2_score, explained_variance_score
)

# Visualization and utilities
from IPython.display import display, HTML
import time
import joblib

print("🤖 Machine Learning Libraries Loaded Successfully!")
print(f"📊 XGBoost Available: {XGBOOST_AVAILABLE}")
print(f"📊 LightGBM Available: {LIGHTGBM_AVAILABLE}")
print("🚀 Ready for Comprehensive Model Comparison!")

In [ ]:
# Load and prepare the dataset
print("📥 Loading Ethiopian Salary Dataset for Model Comparison")
print("=" * 60)

# Load data
df = pd.read_csv('../ethiopia_salary_data.csv')
print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# Prepare features and target
# Handle missing values
df['test_score'].fillna(df['test_score'].median(), inplace=True)

# Define features and target
feature_columns = ['experience_years', 'test_score', 'department', 'education_level', 'location']
target_column = 'salary_etb'

X = df[feature_columns].copy()
y = df[target_column].copy()

print(f"✅ Features prepared: {X.shape[1]} columns")
print(f"✅ Target variable: {target_column}")
print(f"📊 Target range: {y.min():,.0f} - {y.max():,.0f} ETB")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=pd.cut(y, bins=5)
)

print(f"\n📊 Data Split:")
print(f"   Training set: {X_train.shape[0]} samples")
print(f"   Testing set: {X_test.shape[0]} samples")
print(f"   Split ratio: 80% train, 20% test")

## 🔧 Data Preprocessing Pipeline

Create a robust preprocessing pipeline to handle both numerical and categorical features.

In [ ]:
# Create preprocessing pipeline
print("🔧 Creating Data Preprocessing Pipeline")
print("-" * 45)

# Identify numerical and categorical columns
numerical_features = ['experience_years', 'test_score']
categorical_features = ['department', 'education_level', 'location']

print(f"📊 Numerical features: {numerical_features}")
print(f"🏷️  Categorical features: {categorical_features}")

# Create preprocessing steps
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Numerical preprocessing: scaling
numerical_transformer = StandardScaler()

# Categorical preprocessing: one-hot encoding
categorical_transformer = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("✅ Preprocessing pipeline created successfully!")

# Fit and transform the training data to see the final feature count
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\n📊 Processed Features:")
print(f"   Original features: {X_train.shape[1]}")
print(f"   After preprocessing: {X_train_processed.shape[1]}")
print(f"   Feature expansion: {X_train_processed.shape[1] - X_train.shape[1]} new features")

# Get feature names after preprocessing
feature_names = (numerical_features + 
                list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)))

print(f"\n🏷️  Final feature names:")
for i, name in enumerate(feature_names[:10]):  # Show first 10
    print(f"   {i+1:2d}. {name}")
if len(feature_names) > 10:
    print(f"   ... and {len(feature_names) - 10} more features")